# Azure Databricks Final Assessment 2

**Domain:** Retail Supply Chain Analytics

## Objective

Build an end-to-end Databricks pipeline using:

* PySpark DataFrames
* Spark SQL
* Delta Lake
* CRUD
* MERGE / UPSERT
* History
* Time Travel
* VACUUM
* Parquet to Delta
* Incremental Load
* DLT
* Unity Catalog
* Governance


In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Azure Assessment 2").getOrCreate()

---

# Part 0 — Prepare Bigger Dataset

Students can run this first in a normal Python notebook.

---

## products_data

```python
products_data = [

(101,"Rice Bag","Groceries","Hyderabad",1200,50),

(102,"Wheat Flour","Groceries","Bengaluru",900,80),

(103,"Sunflower Oil","Groceries","Mumbai",1800,40),

(104,"Milk Pack","Dairy","Chennai",60,200),

(105,"Cheese Block","Dairy","Delhi",450,70),

(106,"Soap","Personal Care","Kolkata",120,300),

(107,"Shampoo","Personal Care","Pune",320,150),

(108,"Toothpaste","Personal Care","Ahmedabad",90,250),

(109,"Notebook","Stationery","Hyderabad",75,500),

(110,"Pen Pack","Stationery","Mumbai",110,400),

(111,"LED TV","Electronics","Delhi",45000,15),

(112,"Refrigerator","Electronics","Chennai",38000,10),

(113,"Washing Machine","Electronics","Bengaluru",29000,12),

(114,"Mobile Phone","Electronics","Hyderabad",25000,35),

(115,"Laptop","Electronics","Pune",62000,18),

(116,"Air Conditioner","Electronics","Mumbai",42000,9),

(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),

(118,"Water Purifier","Home Appliances","Delhi",12000,20),

(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),

(120,"Gas Stove","Home Appliances","Chennai",5500,25)

]

products_columns = [

"product_id",

"product_name",

"category",

"warehouse_city",

"price",

"stock_quantity"

]

products_df = spark.createDataFrame(products_data, products_columns)
```

---

## suppliers_data

```python
suppliers_data = [

(201,"Reddy Traders","Hyderabad","Groceries"),

(202,"Fresh Dairy Ltd","Chennai","Dairy"),

(203,"CarePlus Suppliers","Mumbai","Personal Care"),

(204,"Elite Electronics","Delhi","Electronics"),

(205,"OfficeKart","Bengaluru","Stationery"),

(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),

(207,"National Grocers","Ahmedabad","Groceries"),

(208,"Smart Electronics","Kolkata","Electronics"),

(209,"Daily Essentials","Hyderabad","Personal Care"),

(210,"Kitchen World","Chennai","Home Appliances")

]

suppliers_columns = [

"supplier_id",

"supplier_name",

"supplier_city",

"specialization"

]

suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)
```

---

## orders_data

```python
orders_data = [

(301,101,201,"2024-04-01",20,"Delivered"),

(302,102,201,"2024-04-01",35,"Delivered"),

(303,111,204,"2024-04-02",2,"Delivered"),

(304,114,208,"2024-04-02",5,"Pending"),

(305,115,204,"2024-04-03",3,"Delivered"),

(306,104,202,"2024-04-03",50,"Delivered"),

(307,105,202,"2024-04-04",18,"Cancelled"),

(308,117,206,"2024-04-04",7,"Delivered"),

(309,118,210,"2024-04-05",4,"Pending"),

(310,119,206,"2024-04-05",12,"Delivered"),

(311,120,210,"2024-04-06",6,"Delivered"),

(312,113,204,"2024-04-06",4,"Delivered"),

(313,116,208,"2024-04-07",2,"Pending"),

(314,109,205,"2024-04-07",80,"Delivered"),

(315,110,205,"2024-04-08",120,"Delivered"),

(316,106,203,"2024-04-08",60,"Cancelled"),

(317,107,209,"2024-04-09",25,"Delivered"),

(318,108,203,"2024-04-09",40,"Delivered"),

(319,112,208,"2024-04-10",2,"Pending"),

(320,101,207,"2024-04-10",15,"Delivered")

]

orders_columns = [

"order_id",

"product_id",

"supplier_id",

"order_date",

"quantity",

"order_status"

]

orders_df = spark.createDataFrame(orders_data, orders_columns)
```

---

## payments_data

```python
payments_data = [

(401,301,24000,"UPI","Paid"),

(402,302,31500,"Credit Card","Paid"),

(403,303,90000,"Bank Transfer","Paid"),

(404,304,125000,"UPI","Pending"),

(405,305,186000,"Bank Transfer","Paid"),

(406,306,3000,"Cash","Paid"),

(407,307,8100,"UPI","Cancelled"),

(408,308,24500,"Debit Card","Paid"),

(409,309,48000,"UPI","Pending"),

(410,310,33600,"Cash","Paid"),

(411,311,33000,"Credit Card","Paid"),

(412,312,116000,"Bank Transfer","Paid"),

(413,313,84000,"UPI","Pending"),

(414,314,6000,"Cash","Paid"),

(415,315,13200,"UPI","Paid"),

(416,316,7200,"Cash","Cancelled"),

(417,317,8000,"UPI","Paid"),

(418,318,3600,"Debit Card","Paid"),

(419,319,76000,"Bank Transfer","Pending"),

(420,320,18000,"UPI","Paid")

]

payments_columns = [

"payment_id",

"order_id",

"bill_amount",

"payment_mode",

"payment_status"

]

payments_df = spark.createDataFrame(payments_data, payments_columns)
```

---


In [0]:
products_data = [

(101,"Rice Bag","Groceries","Hyderabad",1200,50),

(102,"Wheat Flour","Groceries","Bengaluru",900,80),

(103,"Sunflower Oil","Groceries","Mumbai",1800,40),

(104,"Milk Pack","Dairy","Chennai",60,200),

(105,"Cheese Block","Dairy","Delhi",450,70),

(106,"Soap","Personal Care","Kolkata",120,300),

(107,"Shampoo","Personal Care","Pune",320,150),

(108,"Toothpaste","Personal Care","Ahmedabad",90,250),

(109,"Notebook","Stationery","Hyderabad",75,500),

(110,"Pen Pack","Stationery","Mumbai",110,400),

(111,"LED TV","Electronics","Delhi",45000,15),

(112,"Refrigerator","Electronics","Chennai",38000,10),

(113,"Washing Machine","Electronics","Bengaluru",29000,12),

(114,"Mobile Phone","Electronics","Hyderabad",25000,35),

(115,"Laptop","Electronics","Pune",62000,18),

(116,"Air Conditioner","Electronics","Mumbai",42000,9),

(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),

(118,"Water Purifier","Home Appliances","Delhi",12000,20),

(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),

(120,"Gas Stove","Home Appliances","Chennai",5500,25)

]

products_columns = [

"product_id",

"product_name",

"category",

"warehouse_city",

"price",

"stock_quantity"

]

products_df = spark.createDataFrame(products_data, products_columns)

In [0]:
suppliers_data = [

(201,"Reddy Traders","Hyderabad","Groceries"),

(202,"Fresh Dairy Ltd","Chennai","Dairy"),

(203,"CarePlus Suppliers","Mumbai","Personal Care"),

(204,"Elite Electronics","Delhi","Electronics"),

(205,"OfficeKart","Bengaluru","Stationery"),

(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),

(207,"National Grocers","Ahmedabad","Groceries"),

(208,"Smart Electronics","Kolkata","Electronics"),

(209,"Daily Essentials","Hyderabad","Personal Care"),

(210,"Kitchen World","Chennai","Home Appliances")

]

suppliers_columns = [

"supplier_id",

"supplier_name",

"supplier_city",

"specialization"

]

suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

In [0]:
orders_data = [

(301,101,201,"2024-04-01",20,"Delivered"),

(302,102,201,"2024-04-01",35,"Delivered"),

(303,111,204,"2024-04-02",2,"Delivered"),

(304,114,208,"2024-04-02",5,"Pending"),

(305,115,204,"2024-04-03",3,"Delivered"),

(306,104,202,"2024-04-03",50,"Delivered"),

(307,105,202,"2024-04-04",18,"Cancelled"),

(308,117,206,"2024-04-04",7,"Delivered"),

(309,118,210,"2024-04-05",4,"Pending"),

(310,119,206,"2024-04-05",12,"Delivered"),

(311,120,210,"2024-04-06",6,"Delivered"),

(312,113,204,"2024-04-06",4,"Delivered"),

(313,116,208,"2024-04-07",2,"Pending"),

(314,109,205,"2024-04-07",80,"Delivered"),

(315,110,205,"2024-04-08",120,"Delivered"),

(316,106,203,"2024-04-08",60,"Cancelled"),

(317,107,209,"2024-04-09",25,"Delivered"),

(318,108,203,"2024-04-09",40,"Delivered"),

(319,112,208,"2024-04-10",2,"Pending"),

(320,101,207,"2024-04-10",15,"Delivered")

]

orders_columns = [

"order_id",

"product_id",

"supplier_id",

"order_date",

"quantity",

"order_status"

]

orders_df = spark.createDataFrame(orders_data, orders_columns)

In [0]:
payments_data = [

(401,301,24000,"UPI","Paid"),

(402,302,31500,"Credit Card","Paid"),

(403,303,90000,"Bank Transfer","Paid"),

(404,304,125000,"UPI","Pending"),

(405,305,186000,"Bank Transfer","Paid"),

(406,306,3000,"Cash","Paid"),

(407,307,8100,"UPI","Cancelled"),

(408,308,24500,"Debit Card","Paid"),

(409,309,48000,"UPI","Pending"),

(410,310,33600,"Cash","Paid"),

(411,311,33000,"Credit Card","Paid"),

(412,312,116000,"Bank Transfer","Paid"),

(413,313,84000,"UPI","Pending"),

(414,314,6000,"Cash","Paid"),

(415,315,13200,"UPI","Paid"),

(416,316,7200,"Cash","Cancelled"),

(417,317,8000,"UPI","Paid"),

(418,318,3600,"Debit Card","Paid"),

(419,319,76000,"Bank Transfer","Pending"),

(420,320,18000,"UPI","Paid")

]

payments_columns = [

"payment_id",

"order_id",

"bill_amount",

"payment_mode",

"payment_status"

]

payments_df = spark.createDataFrame(payments_data, payments_columns)

---

# Assessment Tasks

# Part 1 — DataFrame Fundamentals

1. Display all DataFrames.
2. Print schema for each DataFrame.
3. Count total records in each DataFrame.
4. Display first 10 products.
5. Display product name, category, and stock quantity.
6. Display suppliers from Hyderabad and Chennai.
7. Display orders with status Delivered.
8. Display pending orders.
9. Display electronics products only.
10. Display products with stock quantity below 20.

---

In [0]:
products_df.show()

+----------+---------------+---------------+--------------+-----+--------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|
+----------+---------------+---------------+--------------+-----+--------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|
|       108|     Toothpaste|  Personal Care|     Ahmedabad|   90|           250|
|       109|       Notebook|     Stationery|     Hyderabad|   75|           500|
|       110|       Pen Pack|

In [0]:
suppliers_df.show()

+-----------+------------------+-------------+---------------+
|supplier_id|     supplier_name|supplier_city| specialization|
+-----------+------------------+-------------+---------------+
|        201|     Reddy Traders|    Hyderabad|      Groceries|
|        202|   Fresh Dairy Ltd|      Chennai|          Dairy|
|        203|CarePlus Suppliers|       Mumbai|  Personal Care|
|        204| Elite Electronics|        Delhi|    Electronics|
|        205|        OfficeKart|    Bengaluru|     Stationery|
|        206| HomeNeeds Pvt Ltd|         Pune|Home Appliances|
|        207|  National Grocers|    Ahmedabad|      Groceries|
|        208| Smart Electronics|      Kolkata|    Electronics|
|        209|  Daily Essentials|    Hyderabad|  Personal Care|
|        210|     Kitchen World|      Chennai|Home Appliances|
+-----------+------------------+-------------+---------------+



In [0]:
orders_df.show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|
|     302|       102|        201|2024-04-01|      35|   Delivered|
|     303|       111|        204|2024-04-02|       2|   Delivered|
|     304|       114|        208|2024-04-02|       5|     Pending|
|     305|       115|        204|2024-04-03|       3|   Delivered|
|     306|       104|        202|2024-04-03|      50|   Delivered|
|     307|       105|        202|2024-04-04|      18|   Cancelled|
|     308|       117|        206|2024-04-04|       7|   Delivered|
|     309|       118|        210|2024-04-05|       4|     Pending|
|     310|       119|        206|2024-04-05|      12|   Delivered|
|     311|       120|        210|2024-04-06|       6|   Delivered|
|     312|       113|        204|2024-04-06|       4|   Delive

In [0]:
payments_df.show()

+----------+--------+-----------+-------------+--------------+
|payment_id|order_id|bill_amount| payment_mode|payment_status|
+----------+--------+-----------+-------------+--------------+
|       401|     301|      24000|          UPI|          Paid|
|       402|     302|      31500|  Credit Card|          Paid|
|       403|     303|      90000|Bank Transfer|          Paid|
|       404|     304|     125000|          UPI|       Pending|
|       405|     305|     186000|Bank Transfer|          Paid|
|       406|     306|       3000|         Cash|          Paid|
|       407|     307|       8100|          UPI|     Cancelled|
|       408|     308|      24500|   Debit Card|          Paid|
|       409|     309|      48000|          UPI|       Pending|
|       410|     310|      33600|         Cash|          Paid|
|       411|     311|      33000|  Credit Card|          Paid|
|       412|     312|     116000|Bank Transfer|          Paid|
|       413|     313|      84000|          UPI|       P

In [0]:
products_df.printSchema()
suppliers_df.printSchema()
orders_df.printSchema()
payments_df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- warehouse_city: string (nullable = true)
 |-- price: long (nullable = true)
 |-- stock_quantity: long (nullable = true)

root
 |-- supplier_id: long (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- supplier_city: string (nullable = true)
 |-- specialization: string (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- supplier_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- bill_amount: long (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [0]:
products_df.count()


20

In [0]:
suppliers_df.count()


10

In [0]:
orders_df.count()


20

In [0]:
payments_df.count()

20

In [0]:
products_df.show(10)

+----------+-------------+-------------+--------------+-----+--------------+
|product_id| product_name|     category|warehouse_city|price|stock_quantity|
+----------+-------------+-------------+--------------+-----+--------------+
|       101|     Rice Bag|    Groceries|     Hyderabad| 1200|            50|
|       102|  Wheat Flour|    Groceries|     Bengaluru|  900|            80|
|       103|Sunflower Oil|    Groceries|        Mumbai| 1800|            40|
|       104|    Milk Pack|        Dairy|       Chennai|   60|           200|
|       105| Cheese Block|        Dairy|         Delhi|  450|            70|
|       106|         Soap|Personal Care|       Kolkata|  120|           300|
|       107|      Shampoo|Personal Care|          Pune|  320|           150|
|       108|   Toothpaste|Personal Care|     Ahmedabad|   90|           250|
|       109|     Notebook|   Stationery|     Hyderabad|   75|           500|
|       110|     Pen Pack|   Stationery|        Mumbai|  110|           400|

In [0]:
products_df.select(products_df.product_name, products_df.category, products_df.stock_quantity).show()

+---------------+---------------+--------------+
|   product_name|       category|stock_quantity|
+---------------+---------------+--------------+
|       Rice Bag|      Groceries|            50|
|    Wheat Flour|      Groceries|            80|
|  Sunflower Oil|      Groceries|            40|
|      Milk Pack|          Dairy|           200|
|   Cheese Block|          Dairy|            70|
|           Soap|  Personal Care|           300|
|        Shampoo|  Personal Care|           150|
|     Toothpaste|  Personal Care|           250|
|       Notebook|     Stationery|           500|
|       Pen Pack|     Stationery|           400|
|         LED TV|    Electronics|            15|
|   Refrigerator|    Electronics|            10|
|Washing Machine|    Electronics|            12|
|   Mobile Phone|    Electronics|            35|
|         Laptop|    Electronics|            18|
|Air Conditioner|    Electronics|             9|
|  Mixer Grinder|Home Appliances|            45|
| Water Purifier|Hom

In [0]:
suppliers_df.filter(suppliers_df.supplier_city.isin("Hyderabad", "Chennai")).show()

+-----------+----------------+-------------+---------------+
|supplier_id|   supplier_name|supplier_city| specialization|
+-----------+----------------+-------------+---------------+
|        201|   Reddy Traders|    Hyderabad|      Groceries|
|        202| Fresh Dairy Ltd|      Chennai|          Dairy|
|        209|Daily Essentials|    Hyderabad|  Personal Care|
|        210|   Kitchen World|      Chennai|Home Appliances|
+-----------+----------------+-------------+---------------+



In [0]:
orders_df.filter(orders_df.order_status == "Delivered").show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|
|     302|       102|        201|2024-04-01|      35|   Delivered|
|     303|       111|        204|2024-04-02|       2|   Delivered|
|     305|       115|        204|2024-04-03|       3|   Delivered|
|     306|       104|        202|2024-04-03|      50|   Delivered|
|     308|       117|        206|2024-04-04|       7|   Delivered|
|     310|       119|        206|2024-04-05|      12|   Delivered|
|     311|       120|        210|2024-04-06|       6|   Delivered|
|     312|       113|        204|2024-04-06|       4|   Delivered|
|     314|       109|        205|2024-04-07|      80|   Delivered|
|     315|       110|        205|2024-04-08|     120|   Delivered|
|     317|       107|        209|2024-04-09|      25|   Delive

In [0]:
orders_df.filter(orders_df.order_status == "Pending").show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     304|       114|        208|2024-04-02|       5|     Pending|
|     309|       118|        210|2024-04-05|       4|     Pending|
|     313|       116|        208|2024-04-07|       2|     Pending|
|     319|       112|        208|2024-04-10|       2|     Pending|
+--------+----------+-----------+----------+--------+------------+



In [0]:
products_df.filter(products_df.category == "Electronics").show()

+----------+---------------+-----------+--------------+-----+--------------+
|product_id|   product_name|   category|warehouse_city|price|stock_quantity|
+----------+---------------+-----------+--------------+-----+--------------+
|       111|         LED TV|Electronics|         Delhi|45000|            15|
|       112|   Refrigerator|Electronics|       Chennai|38000|            10|
|       113|Washing Machine|Electronics|     Bengaluru|29000|            12|
|       114|   Mobile Phone|Electronics|     Hyderabad|25000|            35|
|       115|         Laptop|Electronics|          Pune|62000|            18|
|       116|Air Conditioner|Electronics|        Mumbai|42000|             9|
+----------+---------------+-----------+--------------+-----+--------------+



In [0]:
products_df.filter(products_df.stock_quantity < 20).show()

+----------+---------------+-----------+--------------+-----+--------------+
|product_id|   product_name|   category|warehouse_city|price|stock_quantity|
+----------+---------------+-----------+--------------+-----+--------------+
|       111|         LED TV|Electronics|         Delhi|45000|            15|
|       112|   Refrigerator|Electronics|       Chennai|38000|            10|
|       113|Washing Machine|Electronics|     Bengaluru|29000|            12|
|       115|         Laptop|Electronics|          Pune|62000|            18|
|       116|Air Conditioner|Electronics|        Mumbai|42000|             9|
+----------+---------------+-----------+--------------+-----+--------------+



---

# Part 2 — DataFrame Transformations

11. Convert order_date into date type.
12. Add total_order_value column.
13. Create stock_status column.
14. Create order_priority column.
15. Create expensive_product_flag column.
16. Convert category names into uppercase.
17. Rename warehouse_city to inventory_city.
18. Create inventory_value column.
19. Drop temporary columns.
20. Create low_stock_flag column.

---

In [0]:
from pyspark.sql.functions import *

In [0]:
orders_df.withColumn("order_date", to_date(orders_df.order_date)).show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|
|     302|       102|        201|2024-04-01|      35|   Delivered|
|     303|       111|        204|2024-04-02|       2|   Delivered|
|     304|       114|        208|2024-04-02|       5|     Pending|
|     305|       115|        204|2024-04-03|       3|   Delivered|
|     306|       104|        202|2024-04-03|      50|   Delivered|
|     307|       105|        202|2024-04-04|      18|   Cancelled|
|     308|       117|        206|2024-04-04|       7|   Delivered|
|     309|       118|        210|2024-04-05|       4|     Pending|
|     310|       119|        206|2024-04-05|      12|   Delivered|
|     311|       120|        210|2024-04-06|       6|   Delivered|
|     312|       113|        204|2024-04-06|       4|   Delive

In [0]:
orders_df.join(products_df, "product_id").withColumn("total_order_value", orders_df.quantity * products_df.price).show()

+----------+--------+-----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+-----------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|   product_name|       category|warehouse_city|price|stock_quantity|total_order_value|
+----------+--------+-----------+----------+--------+------------+---------------+---------------+--------------+-----+--------------+-----------------+
|       101|     320|        207|2024-04-10|      15|   Delivered|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|            18000|
|       102|     302|        201|2024-04-01|      35|   Delivered|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|            31500|
|       101|     301|        201|2024-04-01|      20|   Delivered|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|            24000|
|       104|     306|        202|2024-04-03|      50|   Delivered|      Milk Pack|

In [0]:
products_df.withColumn("stock_status",when(col("stock_quantity")<20,"Low").otherwise("Sufficient")).show()

+----------+---------------+---------------+--------------+-----+--------------+------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|stock_status|
+----------+---------------+---------------+--------------+-----+--------------+------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|  Sufficient|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|  Sufficient|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|  Sufficient|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|  Sufficient|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|  Sufficient|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|  Sufficient|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|  Sufficient|
|       108|     Toothpaste|  Personal Care|     Ahmedabad| 

In [0]:
orders_df.withColumn("order_priority",when(col("quantity")>20,"High").when(col("quantity")>=10,"Medium").otherwise("Low")).show()

+--------+----------+-----------+----------+--------+------------+--------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|order_priority|
+--------+----------+-----------+----------+--------+------------+--------------+
|     301|       101|        201|2024-04-01|      20|   Delivered|        Medium|
|     302|       102|        201|2024-04-01|      35|   Delivered|          High|
|     303|       111|        204|2024-04-02|       2|   Delivered|           Low|
|     304|       114|        208|2024-04-02|       5|     Pending|           Low|
|     305|       115|        204|2024-04-03|       3|   Delivered|           Low|
|     306|       104|        202|2024-04-03|      50|   Delivered|          High|
|     307|       105|        202|2024-04-04|      18|   Cancelled|        Medium|
|     308|       117|        206|2024-04-04|       7|   Delivered|           Low|
|     309|       118|        210|2024-04-05|       4|     Pending|           Low|
|     310|      

In [0]:
products_df.withColumn("expensive_product_flag",when(col("price")>20000,"Yes").otherwise("No")).show()

+----------+---------------+---------------+--------------+-----+--------------+----------------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|expensive_product_flag|
+----------+---------------+---------------+--------------+-----+--------------+----------------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|                    No|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|                    No|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|                    No|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|                    No|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|                    No|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|                    No|
|       107|        Shampoo|  Personal Care|          Pune|  320

In [0]:
products_df.withColumn("category",upper(col("category"))).show()

+----------+---------------+---------------+--------------+-----+--------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|
+----------+---------------+---------------+--------------+-----+--------------+
|       101|       Rice Bag|      GROCERIES|     Hyderabad| 1200|            50|
|       102|    Wheat Flour|      GROCERIES|     Bengaluru|  900|            80|
|       103|  Sunflower Oil|      GROCERIES|        Mumbai| 1800|            40|
|       104|      Milk Pack|          DAIRY|       Chennai|   60|           200|
|       105|   Cheese Block|          DAIRY|         Delhi|  450|            70|
|       106|           Soap|  PERSONAL CARE|       Kolkata|  120|           300|
|       107|        Shampoo|  PERSONAL CARE|          Pune|  320|           150|
|       108|     Toothpaste|  PERSONAL CARE|     Ahmedabad|   90|           250|
|       109|       Notebook|     STATIONERY|     Hyderabad|   75|           500|
|       110|       Pen Pack|

In [0]:
products_df.withColumnRenamed("warehouse_city","inventory_city").show()

+----------+---------------+---------------+--------------+-----+--------------+
|product_id|   product_name|       category|inventory_city|price|stock_quantity|
+----------+---------------+---------------+--------------+-----+--------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|
|       108|     Toothpaste|  Personal Care|     Ahmedabad|   90|           250|
|       109|       Notebook|     Stationery|     Hyderabad|   75|           500|
|       110|       Pen Pack|

In [0]:
products_df.withColumn("inventory_value",col("price")*col("stock_quantity")).show()

+----------+---------------+---------------+--------------+-----+--------------+---------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|inventory_value|
+----------+---------------+---------------+--------------+-----+--------------+---------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|          60000|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|          72000|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|          72000|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|          12000|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|          31500|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|          36000|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|          48000|
|       108|     Toothpaste|  

In [0]:
orders_df.join(products_df,"product_id").drop("price").show()

+----------+--------+-----------+----------+--------+------------+---------------+---------------+--------------+--------------+
|product_id|order_id|supplier_id|order_date|quantity|order_status|   product_name|       category|warehouse_city|stock_quantity|
+----------+--------+-----------+----------+--------+------------+---------------+---------------+--------------+--------------+
|       101|     320|        207|2024-04-10|      15|   Delivered|       Rice Bag|      Groceries|     Hyderabad|            50|
|       102|     302|        201|2024-04-01|      35|   Delivered|    Wheat Flour|      Groceries|     Bengaluru|            80|
|       101|     301|        201|2024-04-01|      20|   Delivered|       Rice Bag|      Groceries|     Hyderabad|            50|
|       104|     306|        202|2024-04-03|      50|   Delivered|      Milk Pack|          Dairy|       Chennai|           200|
|       105|     307|        202|2024-04-04|      18|   Cancelled|   Cheese Block|          Dairy

In [0]:
products_df.withColumn("low_stock_flag",when(col("stock_quantity")<20,"Yes").otherwise("No")).show()

+----------+---------------+---------------+--------------+-----+--------------+--------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|low_stock_flag|
+----------+---------------+---------------+--------------+-----+--------------+--------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|            No|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|            No|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|            No|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|            No|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|            No|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|            No|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|            No|
|       108|     Toothpaste|  Personal C

---

# Part 3 — Joins

21. Join products with orders.
22. Join orders with suppliers.
23. Join orders with payments.
24. Create final joined DataFrame.
25. Display product name, supplier name, quantity, bill amount.
26. Find suppliers serving different cities.
27. Find delivered orders with paid payments.
28. Find pending orders with pending payments.
29. Find cancelled orders with cancelled payments.
30. Find products ordered multiple times.

---

In [0]:
from pyspark.sql.functions import *

products_df.join(orders_df,"product_id").show()
orders_df.join(suppliers_df,"supplier_id").show()
orders_df.join(payments_df,"order_id").show()
final_df=products_df.join(orders_df,"product_id").join(suppliers_df,"supplier_id").join(payments_df,"order_id")
final_df.select("product_name","supplier_name","quantity","bill_amount").show()
suppliers_df.groupBy("supplier_name").agg(countDistinct("supplier_city")).show()
final_df.filter(col("order_status")=="Delivered").filter(col("payment_status")=="Paid").show()
final_df.filter(col("order_status")=="Pending").filter(col("payment_status")=="Pending").show()
final_df.filter(col("order_status")=="Cancelled").filter(col("payment_status")=="Cancelled").show()
final_df.groupBy("product_id").count().filter(col("count")>1).show()

+----------+---------------+---------------+--------------+-----+--------------+--------+-----------+----------+--------+------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|order_id|supplier_id|order_date|quantity|order_status|
+----------+---------------+---------------+--------------+-----+--------------+--------+-----------+----------+--------+------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|     320|        207|2024-04-10|      15|   Delivered|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|     302|        201|2024-04-01|      35|   Delivered|
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|     301|        201|2024-04-01|      20|   Delivered|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|     306|        202|2024-04-03|      50|   Delivered|
|       105|   Cheese Block|          Dairy|         De

---

# Part 4 — Aggregations

31. Count products by category.
32. Count orders by status.
33. Count suppliers by city.
34. Calculate total revenue.
35. Calculate average bill amount.
36. Calculate total revenue by category.
37. Calculate total revenue by supplier.
38. Calculate total revenue by city.
39. Find top-selling products.
40. Find lowest-selling products.

---

In [0]:
products_df.groupBy("category").count().show()


+---------------+-----+
|       category|count|
+---------------+-----+
|      Groceries|    3|
|          Dairy|    2|
|  Personal Care|    3|
|     Stationery|    2|
|    Electronics|    6|
|Home Appliances|    4|
+---------------+-----+



In [0]:
orders_df.groupBy("order_status").count().show()


+------------+-----+
|order_status|count|
+------------+-----+
|   Delivered|   14|
|     Pending|    4|
|   Cancelled|    2|
+------------+-----+



In [0]:
suppliers_df.groupBy("supplier_city").count().show()


+-------------+-----+
|supplier_city|count|
+-------------+-----+
|    Hyderabad|    2|
|      Chennai|    2|
|       Mumbai|    1|
|        Delhi|    1|
|    Bengaluru|    1|
|         Pune|    1|
|    Ahmedabad|    1|
|      Kolkata|    1|
+-------------+-----+



In [0]:
final_df.agg(sum("bill_amount")).show()


+----------------+
|sum(bill_amount)|
+----------------+
|          938700|
+----------------+



In [0]:
final_df.agg(avg("bill_amount")).show()


+----------------+
|avg(bill_amount)|
+----------------+
|         46935.0|
+----------------+



In [0]:
final_df.groupBy("category").agg(sum("bill_amount")).show()


+---------------+----------------+
|       category|sum(bill_amount)|
+---------------+----------------+
|      Groceries|           73500|
|    Electronics|          677000|
|          Dairy|           11100|
|Home Appliances|          139100|
|     Stationery|           19200|
|  Personal Care|           18800|
+---------------+----------------+



In [0]:
final_df.groupBy("supplier_name").agg(sum("bill_amount")).show()


+------------------+----------------+
|     supplier_name|sum(bill_amount)|
+------------------+----------------+
|     Reddy Traders|           55500|
| Smart Electronics|          285000|
| Elite Electronics|          392000|
|   Fresh Dairy Ltd|           11100|
| HomeNeeds Pvt Ltd|           58100|
|     Kitchen World|           81000|
|        OfficeKart|           19200|
|CarePlus Suppliers|           10800|
|  Daily Essentials|            8000|
|  National Grocers|           18000|
+------------------+----------------+



In [0]:
final_df.groupBy("supplier_city").agg(sum("bill_amount")).show()


+-------------+----------------+
|supplier_city|sum(bill_amount)|
+-------------+----------------+
|    Hyderabad|           63500|
|        Delhi|          392000|
|      Kolkata|          285000|
|      Chennai|           92100|
|         Pune|           58100|
|    Bengaluru|           19200|
|       Mumbai|           10800|
|    Ahmedabad|           18000|
+-------------+----------------+



In [0]:
final_df.groupBy("product_name").agg(sum("quantity").alias("total_qty")).orderBy(desc("total_qty")).show()


+---------------+---------+
|   product_name|total_qty|
+---------------+---------+
|       Pen Pack|      120|
|       Notebook|       80|
|           Soap|       60|
|      Milk Pack|       50|
|     Toothpaste|       40|
|       Rice Bag|       35|
|    Wheat Flour|       35|
|        Shampoo|       25|
|   Cheese Block|       18|
|    Ceiling Fan|       12|
|  Mixer Grinder|        7|
|      Gas Stove|        6|
|   Mobile Phone|        5|
| Water Purifier|        4|
|Washing Machine|        4|
|         Laptop|        3|
|         LED TV|        2|
|Air Conditioner|        2|
|   Refrigerator|        2|
+---------------+---------+



In [0]:
final_df.groupBy("product_name").agg(sum("quantity").alias("total_qty")).orderBy("total_qty").show()

+---------------+---------+
|   product_name|total_qty|
+---------------+---------+
|         LED TV|        2|
|Air Conditioner|        2|
|   Refrigerator|        2|
|         Laptop|        3|
| Water Purifier|        4|
|Washing Machine|        4|
|   Mobile Phone|        5|
|      Gas Stove|        6|
|  Mixer Grinder|        7|
|    Ceiling Fan|       12|
|   Cheese Block|       18|
|        Shampoo|       25|
|       Rice Bag|       35|
|    Wheat Flour|       35|
|     Toothpaste|       40|
|      Milk Pack|       50|
|           Soap|       60|
|       Notebook|       80|
|       Pen Pack|      120|
+---------------+---------+



---

# Part 5 — Spark SQL

41. Create temp views for all DataFrames.
42. Show all products using SQL.
43. Find electronics orders using SQL.
44. Find revenue by category.
45. Find revenue by supplier.
46. Find top 5 highest bill orders.
47. Count orders per supplier.
48. Count orders per category.
49. Find average payment per mode.
50. Find products generating revenue above 100000.

---

In [0]:
products_df.createOrReplaceTempView("products")
orders_df.createOrReplaceTempView("orders")
suppliers_df.createOrReplaceTempView("suppliers")
payments_df.createOrReplaceTempView("payments")

In [0]:
spark.sql("select * from products").show()
spark.sql("select * from orders where product_id in (select product_id from products where category='Electronics')").show()
spark.sql("select category,sum(price*stock_quantity) as revenue from products group by category").show()
spark.sql("select supplier_name,sum(bill_amount) from suppliers s join orders o on s.supplier_id=o.supplier_id join payments p on o.order_id=p.order_id group by supplier_name").show()
spark.sql("select * from payments order by bill_amount desc limit 5").show()
spark.sql("select supplier_id,count(*) from orders group by supplier_id").show()
spark.sql("select category,count(*) from products group by category").show()
spark.sql("select payment_mode,avg(bill_amount) from payments group by payment_mode").show()
spark.sql("select product_id,sum(bill_amount) from orders o join payments p on o.order_id=p.order_id group by product_id having sum(bill_amount)>100000").show()

+----------+---------------+---------------+--------------+-----+--------------+
|product_id|   product_name|       category|warehouse_city|price|stock_quantity|
+----------+---------------+---------------+--------------+-----+--------------+
|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|
|       102|    Wheat Flour|      Groceries|     Bengaluru|  900|            80|
|       103|  Sunflower Oil|      Groceries|        Mumbai| 1800|            40|
|       104|      Milk Pack|          Dairy|       Chennai|   60|           200|
|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|
|       106|           Soap|  Personal Care|       Kolkata|  120|           300|
|       107|        Shampoo|  Personal Care|          Pune|  320|           150|
|       108|     Toothpaste|  Personal Care|     Ahmedabad|   90|           250|
|       109|       Notebook|     Stationery|     Hyderabad|   75|           500|
|       110|       Pen Pack|

---

# Part 6 — Window Functions

51. Rank products by revenue within category.
52. Rank suppliers by revenue within city.
53. Use ROW_NUMBER to find top product per category.
54. Use DENSE_RANK to rank suppliers by bill amount.
55. Find top 2 suppliers by revenue.
56. Create running total revenue by order date.
57. Create running total revenue by supplier.
58. Rank cities by revenue.
59. Rank categories by revenue.
60. Find highest bill order per payment mode.

---

In [0]:
from pyspark.sql.window import Window
final_df.withColumn("rank",rank().over(Window.partitionBy("category").orderBy(desc("bill_amount")))).show()
final_df.withColumn("rank",rank().over(Window.partitionBy("supplier_city").orderBy(desc("bill_amount")))).show()
final_df.withColumn("row_num",row_number().over(Window.partitionBy("category").orderBy(desc("bill_amount")))).filter(col("row_num")==1).show()
final_df.withColumn("dense_rank",dense_rank().over(Window.orderBy(desc("bill_amount")))).show()
final_df.withColumn("rank",rank().over(Window.orderBy(desc("bill_amount")))).filter(col("rank")<=2).show()
final_df.withColumn("running_total",sum("bill_amount").over(Window.orderBy("order_date"))).show()
final_df.withColumn("running_total_supplier",sum("bill_amount").over(Window.partitionBy("supplier_id").orderBy("order_date"))).show()
final_df.withColumn("rank_city",rank().over(Window.orderBy(desc("bill_amount")))).show()
final_df.withColumn("rank_category",rank().over(Window.partitionBy("category").orderBy(desc("bill_amount")))).show()
final_df.withColumn("rank_mode",rank().over(Window.partitionBy("payment_mode").orderBy(desc("bill_amount")))).filter(col("rank_mode")==1).show()

+--------+-----------+----------+---------------+---------------+--------------+-----+--------------+----------+--------+------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+----+
|order_id|supplier_id|product_id|   product_name|       category|warehouse_city|price|stock_quantity|order_date|quantity|order_status|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|rank|
+--------+-----------+----------+---------------+---------------+--------------+-----+--------------+----------+--------+------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+----+
|     307|        202|       105|   Cheese Block|          Dairy|         Delhi|  450|            70|2024-04-04|      18|   Cancelled|   Fresh Dairy Ltd|      Chennai|          Dairy|       407|       8100|          UPI|     Cancelled|   1|
|     306|        202|       104|   

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-----------+----------+---------------+---------------+--------------+-----+--------------+----------+--------+------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+----------+
|order_id|supplier_id|product_id|   product_name|       category|warehouse_city|price|stock_quantity|order_date|quantity|order_status|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|dense_rank|
+--------+-----------+----------+---------------+---------------+--------------+-----+--------------+----------+--------+------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+----------+
|     305|        204|       115|         Laptop|    Electronics|          Pune|62000|            18|2024-04-03|       3|   Delivered| Elite Electronics|        Delhi|    Electronics|       405|     186000|Bank Transfer|          Paid|         1|
|     304|  

---

# Part 7 — Delta Lake Core

61. Save final joined data as Delta table.
62. Insert new order records.
63. Update one pending order.
64. Update one pending payment.
65. Delete cancelled orders.
66. Create clean_orders Delta table.
67. Query Delta history.
68. Query previous version using Time Travel.
69. Compare old and latest versions.
70. Run VACUUM DRY RUN.

---

In [0]:
final_df=products_df.join(orders_df,"product_id").join(suppliers_df,"supplier_id").join(payments_df,"order_id")

In [0]:
final_df.show()

+--------+-----------+----------+---------------+---------------+--------------+-----+--------------+----------+--------+------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|order_id|supplier_id|product_id|   product_name|       category|warehouse_city|price|stock_quantity|order_date|quantity|order_status|     supplier_name|supplier_city| specialization|payment_id|bill_amount| payment_mode|payment_status|
+--------+-----------+----------+---------------+---------------+--------------+-----+--------------+----------+--------+------------+------------------+-------------+---------------+----------+-----------+-------------+--------------+
|     301|        201|       101|       Rice Bag|      Groceries|     Hyderabad| 1200|            50|2024-04-01|      20|   Delivered|     Reddy Traders|    Hyderabad|      Groceries|       401|      24000|          UPI|          Paid|
|     302|        201|       102|    Wheat Flour|      G

In [0]:
final_df.write.format("delta").mode("overwrite").saveAsTable("final_orders_delta")

In [0]:
%sql
insert into final_orders_delta values
(321, 211, 121, 'Tablet', 'Electronics', 'Bengaluru', 20000, 25, '2024-04-11', 3, 'Pending', 'Tech Supplies', 'Chennai', 'Electronics', 421, 60000, 'UPI', 'Pending'),
(322, 212, 122, 'Desk Chair', 'Furniture', 'Delhi', 5000, 40, '2024-04-11', 5, 'Delivered', 'Office World', 'Delhi', 'Furniture', 422, 25000, 'Cash', 'Paid');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
update final_orders_delta set order_status = 'Cancelled' where order_status="Pending";

num_affected_rows
5


In [0]:
%sql 
update final_orders_delta set payment_status = 'Paid' where payment_status = 'Pending';

num_affected_rows
0


In [0]:
%sql
delete from final_orders_delta where order_status = 'Cancelled';

num_affected_rows
7


In [0]:
%sql
create or replace table clean_orders using delta as
select * from final_orders_delta;

num_affected_rows,num_inserted_rows


In [0]:
%sql
describe history final_orders_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-05-06T09:26:13.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3923362419203449),7cc3d828-7a40-4a74-b4a8-483121a0c59c,0506-083246-bwuppeke-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 6466, p25FileSize -> 6128, numDeletionVectorsRemoved -> 1, minFileSize -> 6128, numAddedFiles -> 1, maxFileSize -> 6128, p75FileSize -> 6128, p50FileSize -> 6128, numAddedBytes -> 6128)",null,Databricks-Runtime/18.1.x-photon-scala2.13
8,2026-05-06T09:26:12.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(order_status#30479 = Cancelled)""])",null,List(3923362419203449),7cc3d828-7a40-4a74-b4a8-483121a0c59c,0506-083246-bwuppeke-v2n,7,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1143, numDeletionVectorsUpdated -> 0, numDeletedRows -> 7, scanTimeMs -> 777, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 365)",null,Databricks-Runtime/18.1.x-photon-scala2.13
7,2026-05-06T09:25:12.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3923362419203449),eda06906-c29b-413d-8d57-384cdeb41bb8,0506-083246-bwuppeke-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 11855, p25FileSize -> 6466, numDeletionVectorsRemoved -> 1, minFileSize -> 6466, numAddedFiles -> 1, maxFileSize -> 6466, p75FileSize -> 6466, p50FileSize -> 6466, numAddedBytes -> 6466)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-06T09:25:10.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(order_status#29452 = Pending)""])",null,List(3923362419203449),eda06906-c29b-413d-8d57-384cdeb41bb8,0506-083246-bwuppeke-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1881, numDeletionVectorsUpdated -> 0, scanTimeMs -> 789, numAddedFiles -> 1, numUpdatedRows -> 5, numAddedBytes -> 5382, rewriteTimeMs -> 1091)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-06T09:24:56.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(payment_status#29330 = Pending)""])",null,List(3923362419203449),e9d57082-afe2-4235-8038-e9adf62d7733,0506-083246-bwuppeke-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 179, numDeletionVectorsUpdated -> 0, scanTimeMs -> 178, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-06T09:24:46.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3923362419203449),d3742e9c-c73c-49ad-946c-94a63e54faa7,0506-083246-bwuppeke-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 10, numRemovedBytes -> 50711, p25FileSize -> 6473, numDeletionVectorsRemoved -> 5, minFileSize -> 6473, numAddedFiles -> 1, maxFileSize -> 6473, p75FileSize -> 6473, p50FileSize -> 6473, numAddedBytes -> 6473)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-06T09:24:45.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,U

In [0]:
%sql
select * from final_orders_delta version as of 1

order_id,supplier_id,product_id,product_name,category,warehouse_city,price,stock_quantity,order_date,quantity,order_status,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status
313,208,116,Air Conditioner,Electronics,Mumbai,42000,9,2024-04-07,2,Pending,Smart Electronics,Kolkata,Electronics,413,84000,UPI,Pending
314,205,109,Notebook,Stationery,Hyderabad,75,500,2024-04-07,80,Delivered,OfficeKart,Bengaluru,Stationery,414,6000,Cash,Paid
315,205,110,Pen Pack,Stationery,Mumbai,110,400,2024-04-08,120,Delivered,OfficeKart,Bengaluru,Stationery,415,13200,UPI,Paid
308,206,117,Mixer Grinder,Home Appliances,Kolkata,3500,45,2024-04-04,7,Delivered,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid
309,210,118,Water Purifier,Home Appliances,Delhi,12000,20,2024-04-05,4,Pending,Kitchen World,Chennai,Home Appliances,409,48000,UPI,Pending
310,206,119,Ceiling Fan,Home Appliances,Ahmedabad,2800,60,2024-04-05,12,Delivered,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid
303,204,111,LED TV,Electronics,Delhi,45000,15,2024-04-02,2,Delivered,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid
304,208,114,Mobile Phone,Electronics,Hyderabad,25000,35,2024-04-02,5,Pending,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending
305,204,115,Laptop,Electronics,Pune,62000,18,2024-04-03,3,Delivered,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid
318,203,108,Toothpaste,Personal Care,Ahmedabad,90,250,2024-04-09,40,Delivered,CarePlus Suppliers,Mumbai,Personal Care,418,3600,Debit Card,Paid


In [0]:
%sql
VACUUM final_orders_delta RETAIN 168 HOURS DRY RUN;

path


---

# Part 8 — Merge / Upsert

Prepare daily update dataset.

```python
daily_orders_data = [

(321,114,204,"2024-04-11",3,"Delivered"),

(322,118,210,"2024-04-11",2,"Delivered"),

(304,114,208,"2024-04-02",5,"Delivered"),

(319,112,208,"2024-04-10",2,"Delivered"),

(323,120,210,"2024-04-12",1,"Pending")

]

daily_orders_columns = [

"order_id",

"product_id",

"supplier_id",

"order_date",

"quantity",

"order_status"

]

daily_orders_df = spark.createDataFrame(daily_orders_data, daily_orders_columns)
```

71. Create Delta target table.
72. Load initial orders.
73. Create temp view for updates.
74. Merge updates into target table.
75. Update existing records.
76. Insert new records.
77. Verify updated order IDs.
78. Verify inserted order IDs.
79. Check Delta history after merge.
80. Explain why MERGE is important.

---


In [0]:
daily_orders_data = [

(321,114,204,"2024-04-11",3,"Delivered"),

(322,118,210,"2024-04-11",2,"Delivered"),

(304,114,208,"2024-04-02",5,"Delivered"),

(319,112,208,"2024-04-10",2,"Delivered"),

(323,120,210,"2024-04-12",1,"Pending")

]

daily_orders_columns = [

"order_id",

"product_id",

"supplier_id",

"order_date",

"quantity",

"order_status"

]

daily_orders_df = spark.createDataFrame(daily_orders_data, daily_orders_columns)

In [0]:
daily_orders_df.show()

+--------+----------+-----------+----------+--------+------------+
|order_id|product_id|supplier_id|order_date|quantity|order_status|
+--------+----------+-----------+----------+--------+------------+
|     321|       114|        204|2024-04-11|       3|   Delivered|
|     322|       118|        210|2024-04-11|       2|   Delivered|
|     304|       114|        208|2024-04-02|       5|   Delivered|
|     319|       112|        208|2024-04-10|       2|   Delivered|
|     323|       120|        210|2024-04-12|       1|     Pending|
+--------+----------+-----------+----------+--------+------------+



In [0]:
daily_orders_df.write.format("delta").mode("overwrite").saveAsTable("daily_orders_df")

In [0]:
%sql
create or replace view source_orders as 
select * from values
(321, 114, 204, '2024-04-11', 5, 'Delivered'),   
(304, 114, 208, '2024-04-02', 6, 'Delivered'),   
(319, 112, 208, '2024-04-10', 2, 'Delivered'),   
(330, 125, 215, '2024-04-13', 4, 'Pending'),
(331, 126, 216, '2024-04-14', 2, 'Delivered')
AS source_orders(order_id,product_id,supplier_id,order_date,quantity,order_status);

In [0]:
%sql
MERGE INTO daily_orders_df AS target
USING source_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
    target.order_status = source.order_status,
    target.quantity = source.quantity

WHEN NOT MATCHED THEN
INSERT (
    order_id,
    product_id,
    supplier_id,
    order_date,
    quantity,
    order_status
)
VALUES (
    source.order_id,
    source.product_id,
    source.supplier_id,
    source.order_date,
    source.quantity,
    source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


In [0]:
%sql 
describe history daily_orders_df;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-05-06T09:41:08.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3923362419203449),f72630f1-c624-49b9-af3b-42d8483a451b,0506-083246-bwuppeke-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 6, numRemovedBytes -> 11301, p25FileSize -> 2140, numDeletionVectorsRemoved -> 1, minFileSize -> 2140, numAddedFiles -> 1, maxFileSize -> 2140, p75FileSize -> 2140, p50FileSize -> 2140, numAddedBytes -> 2140)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-05-06T09:41:07.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(order_id#35682L = cast(order_id#35670 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3923362419203449),f72630f1-c624-49b9-af3b-42d8483a451b,0506-083246-bwuppeke-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 5, numTargetBytesAdded -> 9190, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 2220, materializeSourceTimeMs -> 70, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 882, numTargetRowsUpdated -> 3, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1245)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T09:33:09.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3923362419203449),f077f5a7-e94e-44cf-9ba2-28d7361b0a26,0506-083246-bwuppeke-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2111)",null,Databricks-Runtime/18.1.x-photon-scala2.13


---

# Part 11 — Unity Catalog and Governance

96. Create catalog.
97. Create schema.
98. Register Delta table in Unity Catalog.
99. Create derived revenue table and inspect lineage.
100. Apply SELECT permissions and explain governance behavior.

In [0]:
%sql
create catalog azure_retail_orders_catalog;

In [0]:
%sql
create schema azure_retail_orders_catalog.retail_schema;

In [0]:
%sql
create table azure_retail_orders_catalog.retail_schema.final_orders using delta as select * from final_orders_delta;

num_affected_rows,num_inserted_rows


In [0]:
%sql
create table azure_retail_orders_catalog.retail_schema.revenue_table as select category,sum(bill_amount) as total_revenue from azure_retail_orders_catalog.retail_schema.final_orders group by category;

num_affected_rows,num_inserted_rows


In [0]:
%sql
describe history azure_retail_orders_catalog.retail_schema.revenue_table;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-05-06T10:12:01.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3923362419203449),16e67719-e227-4a5a-a50e-88d8903b8a91,0506-083246-bwuppeke-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 8, numOutputBytes -> 1063)",null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
%sql
grant select on table hexaworksspace.default.final_orders_delta to `azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com`;